### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easliy parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000013298FB0980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000013298FB1400>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000013298FB0980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000013298FB1400>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out 

In [5]:
model.invoke("Provide details about the movie inception.")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie "Inception." Let me start by recalling what I know. I think it\'s a sci-fi movie directed by Christopher Nolan. The main character is Dom Cobb, played by Leonardo DiCaprio. The plot involves dreams, right? There\'s something about entering people\'s dreams to steal information or implant ideas.\n\nI remember there\'s a concept called "inception," which is planting an idea into someone\'s subconscious. The term is used in the movie as the main plot device. The team uses technology to enter dreams, but there are layers of dreams. Maybe there\'s a risk of getting stuck in the dream world? There\'s a character named Ariadne who\'s a dream architect, and another named Eames who can impersonate people. \n\nThe movie has a complex structure with different time layers, where time moves slower in deeper layers of dreams. There\'s a scene with a rotating hallway fight, maybe between Cobb and Zee. The ending is ambiguo

In [6]:
response=model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The movie rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response=model_with_structure.invoke("Provide details about movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me see. I need to use the Movie function they provided. The required parameters are title, year, director, and rating. First, I should confirm the title is "Inception". The director is Christopher Nolan, right? It came out in 2010. The rating... maybe 8.8 on IMDb? Wait, I should check if that\'s accurate. But since I can\'t look it up, I\'ll go with commonly known info. Let me structure the JSON with those details. Make sure all required fields are included. Alright, that should do it.\n', 'tool_calls': [{'id': '977g049f1', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 181, 'prompt_tokens': 229, 'total_tokens': 410, 'completion_time': 0.320136746, 'completion_tokens_details': {'reasoning_tokens

### Nested Structure

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:str
    cast:list[Actor]
    budget:float|None=Field(None,description="Budget in millions USD")

model_with_structure=model.with_structured_output(MovieDetails)

response=model_with_structure.invoke("Provide details about the movie innception")
response

MovieDetails(title='Inception', year='2010', cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito')], budget=160.0)

### TypedDict

TypedDict provides a simpler alternative using Python's built in typing, ideal when you dont need runtime validation.

In [10]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response


{'director': 'Joss Whedon',
 'rating': 7.5,
 'title': 'The Avengers',
 'year': 2012}

In [11]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Avengers',
 'year': 2012}

In [12]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [13]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [17]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result)

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='8c96ec2f-8505-4636-bcfe-d6712fc9026c'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let me see. The user wants me to extract contact information from the given text: "John Doe, john@example.com, (555) 123-4567". The tools provided include a ContactInfo function with parameters for name, email, and phone. All three are required.\n\nFirst, I need to parse the input. The name is John Doe. The email is clearly john@example.com. The phone number is (555) 123-4567. I should check if the phone number is in the correct format. The function expects a string for each parameter, so I don\'t need to change anything there. Let me make sure there are no extra spaces or characters. The name is split into first and last name, but the function just needs the full name as a single string. So John Doe is fine. The email and ph

In [18]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [19]:
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [20]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')